In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *
import re
from datetime import datetime

o_schema = StructType([
    StructField('order_id',IntegerType()),
    StructField('customer_id',IntegerType()),
    StructField('product_id',IntegerType()),
    StructField('quantity',IntegerType()),
    StructField('order_amount',IntegerType()),
    StructField('order_date',StringType()),
    StructField('status',StringType()),
])

files = dbutils.fs.ls('/Volumes/madstagescd2/madstage/madstage/incrementalLoad/')

ordersfiles = []

for file in files:
    match = re.search(r'(\d{4}-\d{2}-\d{2})',file.path)
    if file.name.startswith('orders'):
        if match:
            file_time = datetime.strptime(match.group(1),"%Y-%m-%d")
            ordersfiles.append((file_time,file.path))

if ordersfiles:
    latest_file = max(ordersfiles,key=lambda x:x[0])
    print(latest_file[0],latest_file[1])
    orders = (spark.read.format('csv').option('header','true').schema(o_schema).load(latest_file[1]))

orders = orders.dropna(subset=('customer_id'))
orders = orders.dropDuplicates()
orders.fillna({
    'quantity':0,
    'order_amount':0,
    'status':'Unknown',
})

orders = orders.withColumn('status',f.initcap(f.trim(f.col('status')))).withColumn('order_amount',f.trim('order_amount'))

orders = orders.withColumn('dataValidation',f.when((f.col('order_amount')>=0),'Pass').otherwise('Fail'))

orders = orders.withColumn('is_current',f.lit('Y')).withColumn('updated_dt',f.lit('').cast('string')).withColumn('created_dt',f.current_date().cast('string'))

orders = orders.filter(f.col('dataValidation')=='Pass').drop('dataValidation')

orders.createOrReplaceTempView('orders')
legacy = spark.read.table('madstagescd2.madstage.silver_orders')
orders.createOrReplaceTempView('orders')

spark.sql("""
       merge into madstagescd2.madstage.silver_orders as t
       using orders as s
       on t.order_id = s.order_id
       when matched then
       update set
       t.customer_id = s.customer_id,
       t.product_id = s.product_id,
       t.quantity = s.quantity,
       t.order_amount = s.order_amount,
       t.order_date = s.order_date,
       t.status = s.status,
       t.is_current = s.is_current,
       t.updated_dt = s.updated_dt,
       t.created_dt = s.created_dt
       when not matched then
       insert *       
""")

print('SCD1 implemented successfully')


